# AICIDENTIA - 06 RQ4: similar-incident retrieval

**RQ4:** Can we retrieve similar incidents? Compare TF-IDF vs BM25 vs sentence embeddings (and a hybrid) using precision@5 and MRR against **editor-marked similar incidents** from the snapshot.

> CRISP-DM phases: **Modeling** (retrieval) and **Evaluation** (MRR, nDCG@10, P@5 with bootstrap CIs and paired tests)

Logic lives in `aicidentia.models.retrieval` and `retrieval_outputs`; the write-up is `docs/rq4_similarity.md`. Ground truth comes from AIID editors, so no manual relevance judgements are used. Results show titles only.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from aicidentia import registry
from aicidentia.config import PROJECT_ROOT, configure_logging, load_config, set_seeds

configure_logging()
cfg = load_config()
SEED = set_seeds(cfg["random_seed"])
FIG = PROJECT_ROOT / cfg["paths"]["figures"]
pd.set_option("display.width", 220, "display.max_colwidth", 90)
print(f"{cfg['project_name']} | seed {SEED} | snapshot {cfg['snapshot_date']}")

## 1. Build the index and the ground truth, evaluate every method

Three settings: all incidents as candidates (primary), only earlier incidents (precedent search), and only the link direction the editor recorded.

In [ ]:
from aicidentia.models import retrieval as R
from aicidentia.models import retrieval_outputs

res = retrieval_outputs.run_retrieval()
retrieval_outputs.write_all(res)
f = res.facts
print(
    "indexed:",
    f["indexed_incidents"],
    "| queries:",
    f["queries"],
    "| pairs:",
    f["relevant_pairs"],
    "| oracle:",
    f["oracle"],
)
print("editor links that are among AIID's own top-3 suggestions:", f["aiid_suggestion_overlap"])

## 2. Results (primary setting)

In [ ]:
cols = [
    "method",
    "queries",
    "rr",
    "rr_low",
    "rr_high",
    "ndcg@10",
    "p@5",
    "hit@5",
    "r@10",
    "dissimilar@5",
]
display(res.summary["all"][cols].round(3))
display(res.comparisons["all"].round(4))
display(Image(filename=FIG / "fig_rq4_1_method_comparison.png"))
display(Image(filename=FIG / "fig_rq4_2_recall_at_k.png"))

## 3. Sensitivity: precedent search and directed links

In [ ]:
for setting in ["earlier", "directed"]:
    print(setting)
    display(res.summary[setting][cols].round(3))
    display(
        res.comparisons[setting][
            ["method", "diff", "ci_low", "ci_high", "p_value", "p_holm"]
        ].round(4)
    )
display(res.suggestions.round(3))

## 4. Deployed method and examples (titles only)

In [ ]:
print("deployed:", res.deployed, "|", res.deployed_reason)
display(res.examples[["query", "query_title", "rank", "incident_id", "title", "editor_similar"]])

## 5. Try a free-text query

In [ ]:
print(
    R.search(
        res.index,
        res.deployed,
        "voice clone used to impersonate a company executive in a fraud call",
        k=5,
    )
)

In [ ]:
for key in ["rq4.best_method", "rq4.deployed", "rq4.queries"]:
    print(key, "=", registry.get(key))